# Phase B: merge of PMS and CRM

Input: the two clean files written by Phase A (`pms_cleaning.ipynb` and `crm_cleaning.ipynb`). For each step: what it does, why, and the rows before and after.

- **CHECK** cells show the numbers the next decision is based on. They do not change the data.
- The two systems describe the same bookings, so here we can finally use one file to check and complete the other.
- **Same rules as Phase A:** before changing a value we create a flag column; a fill that is not proven is an ESTIMATE; no invented data.
- The notebook lives in `Project/merge`, reads from `Project/data/processed` and writes the merged table there.
- **Run it with Restart & Run All**, after the two Phase A notebooks. Several cells create a flag and then change the values it marks: run alone a second time, they find nothing left to change and reset the flag.

In [1]:
import pandas as pd

## 1. Load
The clean files of Phase A. `booking_id` (PMS) and `BookingRef` (CRM) are the booking code: before merging we check that each one identifies a single row, otherwise the merge would multiply rows.

In [2]:
pms = pd.read_csv("../data/processed/pms_clean.csv", parse_dates=["record_updated_at"])
crm = pd.read_csv("../data/processed/crm_clean.csv", parse_dates=["checkin_date", "reservation_status_date"])
print("PMS:", pms.shape, "| booking_id unique:", pms["booking_id"].is_unique)
print("CRM:", crm.shape, "| BookingRef unique:", crm["BookingRef"].is_unique)

PMS: (24250, 29) | booking_id unique: True
CRM: (24500, 25) | BookingRef unique: True


## 2. Prepare the merge

### Which columns mean the same thing
The two files use different names. Shared information:

| PMS | CRM | Note |
|---|---|---|
| `booking_id` | `BookingRef` | merge key |
| `hotel` | `property` | same two hotels |
| `arrival_date_year` + `_month` + `_day_of_month` | `checkin_date` | arrival date |
| `adults` | `n_adults` | number of adults |
| `is_canceled` | `reservation_status` | Canceled / No-Show = cancelled, Check-Out = stayed |
| `adr` × nights | `total_price` | price of the stay |
| weekend + week nights | `reservation_status_date` − `checkin_date` | length of stay, only for Check-Out |

### Key and source columns
We give the key the same name in both files, so the merge can use `on`. No other column name is shared, so no suffixes are needed.

We build the PMS arrival date as a real date now, while its three columns are still integers (after an outer merge they would become floats).

We also add `in_pms` and `in_crm`: after an outer merge they tell which file each row comes from.

In [3]:
crm = crm.rename(columns={"BookingRef": "booking_id"})
print("shared column names:", set(pms.columns) & set(crm.columns))

pms["arrival_date"] = pd.to_datetime(pms["arrival_date_year"].astype(str) + "-"
                                     + pms["arrival_date_month"].astype(str) + "-"
                                     + pms["arrival_date_day_of_month"].astype(str), errors="coerce")
print("PMS arrival dates not readable:", pms["arrival_date"].isna().sum())

pms["in_pms"] = True
crm["in_crm"] = True
keys_pms = set(pms["booking_id"])
keys_crm = set(crm["booking_id"])
print("CHECK | in both:", len(keys_pms & keys_crm),
      "| only PMS:", len(keys_pms - keys_crm), "| only CRM:", len(keys_crm - keys_pms))
all_keys = keys_pms | keys_crm
print("CHECK | distinct codes:", len(all_keys), "| first:", min(all_keys), "| last:", max(all_keys))

shared column names: {'booking_id'}
PMS arrival dates not readable: 0
CHECK | in both: 23750 | only PMS: 500 | only CRM: 750
CHECK | distinct codes: 25000 | first: BK300001 | last: BK325000


## 3. Merge

**Decision: outer merge.** 23,750 bookings are in both files, 500 only in the PMS and 750 only in the CRM. Together they are the "about 25,000 bookings" of the brief, which also says that some bookings appear in only one file.
- An inner merge would keep 23,750 rows and lose 1,250 real bookings.
- A left or right merge would lose the bookings of one of the two files.
- The outer merge keeps one row per booking. The columns of the missing file are NaN.

The codes of the two files together go from BK300001 to BK325000 without gaps, each used once (CHECK in section 2), so the 1,250 bookings found in only one file are not explained by codes written wrong.

Expected rows: 23,750 + 500 + 750 = 25,000. We check the shape before and after.

In [4]:
print("rows before: PMS", len(pms), "| CRM", len(crm))
df = pd.merge(pms, crm, on="booking_id", how="outer")
df["in_pms"] = df["in_pms"].notna()
df["in_crm"] = df["in_crm"].notna()
print("rows after:", len(df), "| columns:", df.shape[1], "| booking_id unique:", df["booking_id"].is_unique)
pd.crosstab(df["in_pms"], df["in_crm"])

rows before: PMS 24250 | CRM 24500
rows after: 25000 | columns: 56 | booking_id unique: True


in_crm,False,True
in_pms,,
False,0,750
True,500,23750


### Columns that changed type
With the outer merge, integer columns of one file get NaN on the rows of the other file, so pandas turns them into floats. The flag columns of Phase A become NaN for the rows that were not in that file: there the flag was never checked, so `False` (not touched) is the right value.

In [5]:
flags = ["meal_undefined", "filled_from_old_version", "adr_invalid", "adr_outlier", "adr_zero", "week_nights_missing",
         "zero_nights", "no_guests", "children_missing",
         "segment_undefined", "channel_undefined", "country_placeholder", "country_cn_ambiguous",
         "lead_time_missing", "adults_over_10", "date_inconsistent"]
print("NaN in the flags before:", df[flags].isna().sum().sum())
for col in flags:
    df[col] = df[col] == True
print("NaN in the flags after:", df[flags].isna().sum().sum())
df[flags].sum()

NaN in the flags before: 10250
NaN in the flags after: 0


meal_undefined             236
filled_from_old_version      3
adr_invalid                 70
adr_outlier                  1
adr_zero                   377
week_nights_missing         35
zero_nights                147
no_guests                   31
children_missing           809
segment_undefined            2
channel_undefined            3
country_placeholder        114
country_cn_ambiguous       228
lead_time_missing           25
adults_over_10               2
date_inconsistent           88
dtype: int64

## 4. Shared columns: one value per booking

For each shared piece of information we keep **one** column. The PMS is the reference: it is the hotel management system, and on the shared columns it has no missing values (the CRM has 244, in `n_adults`). The CRM fills it only where the PMS has no value, i.e. the 750 bookings that are only in the CRM.

Conflicts between the two files, where both have a value and they differ, are handled in section 5: here no PMS value is changed.

### 4.1 Hotel

In [6]:
both = df["in_pms"] & df["in_crm"]
print("CHECK | hotel different from property, rows in both files:", (both & (df["hotel"] != df["property"])).sum())
print("hotel missing before:", df["hotel"].isna().sum())
df["hotel"] = df["hotel"].fillna(df["property"])
print("hotel missing after:", df["hotel"].isna().sum())
df["hotel"].value_counts()

CHECK | hotel different from property, rows in both files: 0
hotel missing before: 750
hotel missing after: 0


hotel
City Hotel      16621
Resort Hotel     8379
Name: count, dtype: int64

### 4.2 Arrival date
The PMS date built in section 2 is completed with `checkin_date` for the CRM-only rows. The three PMS columns (year, month, day) stay; for the CRM-only rows they are filled from `arrival_date`. The conflicts are checked in section 5.

In [7]:
print("arrival_date missing before:", df["arrival_date"].isna().sum())
df["arrival_date"] = df["arrival_date"].fillna(df["checkin_date"])
print("arrival_date missing after:", df["arrival_date"].isna().sum(),
      "| from", df["arrival_date"].min().date(), "to", df["arrival_date"].max().date())
parts = ["arrival_date_year", "arrival_date_month", "arrival_date_day_of_month"]
print("year, month, day missing before:", df[parts].isna().sum().sum())
df["arrival_date_year"] = df["arrival_date_year"].fillna(df["arrival_date"].dt.year).astype(int)
df["arrival_date_month"] = df["arrival_date_month"].fillna(df["arrival_date"].dt.month).astype(int)
df["arrival_date_day_of_month"] = df["arrival_date_day_of_month"].fillna(df["arrival_date"].dt.day).astype(int)
print("year, month, day missing after:", df[parts].isna().sum().sum())

arrival_date missing before: 750
arrival_date missing after: 0 | from 2015-07-01 to 2017-08-31
year, month, day missing before: 2250
year, month, day missing after: 0


### 4.3 Adults
`adults` is complete in the PMS; `n_adults` has 244 NaN in the CRM. We complete `adults` with `n_adults` on the CRM-only rows.

In [8]:
print("adults missing before:", df["adults"].isna().sum())
df["adults"] = df["adults"].fillna(df["n_adults"])
print("adults missing after:", df["adults"].isna().sum(),
      "| of which CRM-only rows with n_adults missing:", (~df["in_pms"] & df["n_adults"].isna()).sum())

adults missing before: 750
adults missing after: 8 | of which CRM-only rows with n_adults missing: 8


### 4.4 Cancellation
`reservation_status` says the same thing as `is_canceled`. The crosstab on the rows in both files shows the rule: Canceled and No-Show are 1, Check-Out is 0. The only exceptions are 50 rows, handled in section 5.

In [9]:
status_code = df["reservation_status"].map({"Canceled": 1, "No-Show": 1, "Check-Out": 0})
print(pd.crosstab(df["is_canceled"][both], df["reservation_status"][both]))
print("is_canceled missing before:", df["is_canceled"].isna().sum())
df["is_canceled"] = df["is_canceled"].fillna(status_code)
print("is_canceled missing after:", df["is_canceled"].isna().sum())

reservation_status  Canceled  Check-Out  No-Show
is_canceled                                     
0.0                       50      14841        0
1.0                     8610          0      249
is_canceled missing before: 750
is_canceled missing after: 0


### 4.5 What the outer merge leaves missing
For the 500 PMS-only rows all CRM columns are NaN, and for the 750 CRM-only rows the PMS columns without a CRM counterpart (meal, room types, children, babies, nights, adr, ...) are NaN. Meal, room types, children and babies are in neither file, so they stay missing: nothing is invented.

Nights and adr could be derived for the 485 CRM-only Check-Out rows (472 of them with a price): status date minus check-in, then `total_price / nights` (CHECK below). We do not do it: on the rows in both files, only the PMS reveals a wrong CRM date (5.1, 5.4) or an inflated CRM price (6.1), and without the PMS these errors cannot be seen. So these values stay missing too.

In [10]:
crm_only = df["in_crm"] & ~df["in_pms"]
crm_only_out = crm_only & (df["reservation_status"] == "Check-Out")
print("CHECK | CRM-only rows:", crm_only.sum(), "| Check-Out:", crm_only_out.sum(),
      "| Check-Out with a price:", (crm_only_out & df["total_price"].notna()).sum())
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)

CHECK | CRM-only rows: 750 | Check-Out: 485 | Check-Out with a price: 472


company_id                        23624
agent_id                           3800
children                           1559
meal                               1469
total_price                         990
adr                                 940
stays_in_week_nights                785
required_car_parking_spaces         750
assigned_room_type                  750
booking_changes                     750
deposit_type                        750
record_updated_at                   750
babies                              750
reserved_room_type                  750
stays_in_weekend_nights             750
total_of_special_requests           750
n_adults                            744
guest_country                       614
lead_time_days                      525
distribution_channel                503
market_segment                      502
is_repeated_guest                   500
property                            500
previous_cancellations              500
customer_type                       500


## 5. Conflicts between the two files

Rows in both files where the PMS and the CRM give different values for the same information. The course rules (Sessions 3 and 4):
- assessment finds a conflict, it does not choose;
- **correct only with evidence of the cause**, otherwise keep the value and flag it ("a guess dressed as a fix is worse than a flag").

So for each conflict we look for evidence in a third column (usually the price, which links nights and adr), and we change a value only when the evidence points to one file.

### 5.1 Arrival date
45 bookings have a different arrival date in the two files. For the Check-Out rows we can test which date is right: departure (CRM status date) minus arrival must give the nights of the PMS.

In [11]:
nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
arrival_diff = both & (df["arrival_date"] != df["checkin_date"])
checkout = df["reservation_status"] == "Check-Out"
print("arrival conflicts:", arrival_diff.sum(),
      "| days CRM - PMS:", (df["checkin_date"] - df["arrival_date"])[arrival_diff].dt.days.value_counts().to_dict())
print("CHECK | Check-Out among them:", (arrival_diff & checkout).sum(),
      "| departure - PMS arrival = PMS nights:",
      (arrival_diff & checkout & ((df["reservation_status_date"] - df["arrival_date"]).dt.days == nights)).sum(),
      "| departure - CRM check-in = PMS nights:",
      (arrival_diff & checkout & ((df["reservation_status_date"] - df["checkin_date"]).dt.days == nights)).sum())

arrival conflicts: 45 | days CRM - PMS: {1: 16, -1: 15, 7: 14}
CHECK | Check-Out among them: 30 | departure - PMS arrival = PMS nights: 30 | departure - CRM check-in = PMS nights: 0


**Decision:** the PMS date is right in every row we can test, so `arrival_date` keeps the PMS value (it already does). Flag `arrival_conflict` on the 45 rows; no value changes.

In [12]:
df["arrival_conflict"] = arrival_diff
print("arrival_conflict:", df["arrival_conflict"].sum(), "| rows:", len(df))

arrival_conflict: 45 | rows: 25000


### 5.2 Adults
75 bookings have a different number of adults. They are of two kinds:
- (a) the PMS says more than 10 adults and the CRM 1 or 2;
- (b) the two files differ by one adult.

The price is the independent evidence: a room for 55 people does not cost the same as a room for 2.

In [13]:
adults_diff = both & df["n_adults"].notna() & (df["adults"] != df["n_adults"])
typo = adults_diff & (df["adults"] > 10) & (df["n_adults"] <= 10)
print("adults conflicts:", adults_diff.sum(), "| (a) PMS > 10, CRM <= 10:", typo.sum(),
      "| (b) others:", (adults_diff & ~typo).sum())
print("CHECK (a) | PMS values:", df["adults"][typo].value_counts().to_dict(),
      "| CRM values:", df["n_adults"][typo].value_counts().to_dict())
print("   adr of these rows: min", df["adr"][typo].min(), "max", df["adr"][typo].max(),
      "| median adr by adults (PMS):", df[df["adults"] <= 4].groupby("adults")["adr"].median().to_dict())
print("   reserved room types:", df["reserved_room_type"][typo].value_counts().to_dict())
confirmed = both & (df["adults"] > 10) & (df["adults"] == df["n_adults"])
print("   more than 10 adults confirmed by the CRM:", confirmed.sum(), "| their adr:", df["adr"][confirmed].tolist(),
      "| customer type:", df["customer_type"][confirmed].tolist())
print("   adr > 0 among the (a) rows:", (df["adr"][typo] > 0).sum(), "of", typo.sum())
med = df[df["adults"] <= 4].groupby("adults")["adr"].median()
gap_pms = (df["adr"] - df["adults"].map(med)).abs()
gap_crm = (df["adr"] - df["n_adults"].map(med)).abs()
b_rows = adults_diff & ~typo
print("CHECK (b) | adr closer to the median for the PMS value:", (b_rows & (gap_pms < gap_crm)).sum(),
      "| for the CRM value:", (b_rows & (gap_crm < gap_pms)).sum(),
      "| equal or not testable:", (b_rows & ~(gap_pms < gap_crm) & ~(gap_crm < gap_pms)).sum())
pd.crosstab(df["adults"][adults_diff & ~typo], df["n_adults"][adults_diff & ~typo])

adults conflicts: 75 | (a) PMS > 10, CRM <= 10: 19 | (b) others: 56
CHECK (a) | PMS values: {55.0: 7, 26.0: 6, 20.0: 3, 50.0: 2, 40.0: 1} | CRM values: {2.0: 15, 1.0: 4}
   adr of these rows: min 34.0 max 157.0 | median adr by adults (PMS): {0.0: 9.0, 1.0: 80.0, 2.0: 95.0, 3.0: 151.75, 4.0: 192.86}
   reserved room types: {'A': 16, 'E': 2, 'D': 1}
   more than 10 adults confirmed by the CRM: 2 | their adr: [0.0, 0.0] | customer type: ['Group', 'Group']
   adr > 0 among the (a) rows: 19 of 19
CHECK (b) | adr closer to the median for the PMS value: 32 | for the CRM value: 24 | equal or not testable: 0


n_adults,1.0,2.0,3.0,4.0
adults,,,,
1.0,0,2,0,0
2.0,35,0,15,0
3.0,0,3,0,1


**Decision (a): correct.** The PMS values come from a small fixed set (20, 26, 40, 50, 55), the rooms are mostly type A and the price is that of 1-2 people. The CRM and the price agree, so this is a likely mistake of the PMS: `adults` takes the CRM value, flag `adults_from_crm`. The 2 real groups (26 and 55 adults) are confirmed by the CRM and stay.

**Decision (b): flag only.** A difference of one adult has no evidence either way (the adr is closer to the median price for the PMS value in 32 cases and for the CRM value in 24; CHECK above). The PMS value stays, flag `adults_conflict`.

**More than 10 adults and no CRM value (2 rows): flag `adults_unverified`.** The 21 rows we can check give a clear rule: the 19 wrong values all have adr > 0 (a normal room price), the 2 real groups both have adr 0 and type `Group`. So:
- adr > 0: the value has the profile of the 19 errors, and 20 guests in one room at a normal price is impossible. It becomes NaN (we do not put 2: that would be an ESTIMATE);
- adr 0: the profile of the real groups. The value stays.

In [14]:
df["adults_from_crm"] = typo
df.loc[typo, "adults"] = df.loc[typo, "n_adults"]
df["adults_conflict"] = adults_diff & ~typo
df["adults_unverified"] = (df["adults"] > 10) & df["n_adults"].isna()
impossible = df["adults_unverified"] & (df["adr"] > 0)
print("adults_unverified:", df.loc[df["adults_unverified"], ["booking_id", "adults", "adr"]].values.tolist(),
      "| set to NaN:", impossible.sum())
df["adults"] = df["adults"].where(~impossible)
print("adults_from_crm:", df["adults_from_crm"].sum(), "| adults_conflict:", df["adults_conflict"].sum(),
      "| adults_unverified:", df["adults_unverified"].sum())
print("adults > 10 after:", (df["adults"] > 10).sum(), "| rows:", len(df))

adults_unverified: [['BK305879', 20.0, 77.76], ['BK307085', 20.0, 0.0]] | set to NaN: 1
adults_from_crm: 19 | adults_conflict: 56 | adults_unverified: 2
adults > 10 after: 3 | rows: 25000


### 5.3 Cancellation
50 bookings are not cancelled in the PMS but `Canceled` in the CRM. They are exactly the 50 `date_inconsistent` rows "Canceled with a status date after check-in". If the guest really stayed, the status date should be the departure day: arrival plus nights.

In [15]:
status_diff = both & (df["is_canceled"] == 0) & (df["reservation_status"] == "Canceled")
print("status conflicts:", status_diff.sum(), "| also date_inconsistent:", (status_diff & df["date_inconsistent"]).sum())
print("CHECK | status date = arrival + PMS nights:",
      (status_diff & ((df["reservation_status_date"] - df["arrival_date"]).dt.days == nights)).sum())

status conflicts: 50 | also date_inconsistent: 50
CHECK | status date = arrival + PMS nights: 50


**Decision: correct.** In all 50 rows the status date is exactly the departure day, so the guest stayed and the CRM label is wrong. `is_canceled` stays 0, `reservation_status` becomes `Check-Out`, flag `status_from_pms`. This way the two columns no longer contradict each other.

In [16]:
df["status_from_pms"] = status_diff
df.loc[status_diff, "reservation_status"] = "Check-Out"
print("status_from_pms:", df["status_from_pms"].sum())
pd.crosstab(df["is_canceled"][both], df["reservation_status"][both])

status_from_pms: 50


reservation_status,Canceled,Check-Out,No-Show
is_canceled,,,
0.0,0,14891,0
1.0,8610,0,249


### 5.4 Nights
For Check-Out rows, departure minus arrival must give the PMS nights. Where it does not, the price decides: `total_price / adr` gives the nights the guest paid for.

In [17]:
checkout = df["reservation_status"] == "Check-Out"
stay = (df["reservation_status_date"] - df["arrival_date"]).dt.days
nights_diff = both & checkout & nights.notna() & (stay != nights)
paid_nights = (df["total_price"] / df["adr"]).round()
testable = nights_diff & df["total_price"].notna() & (df["adr"] > 0)
print("nights conflicts:", nights_diff.sum(), "| also date_inconsistent:", (nights_diff & df["date_inconsistent"]).sum())
print("CHECK | testable on the price:", testable.sum(), "| price confirms PMS nights:", (testable & (paid_nights == nights)).sum())

nights conflicts: 35 | also date_inconsistent: 30
CHECK | testable on the price: 32 | price confirms PMS nights: 31


In [18]:
booking_date = df["arrival_date"] - pd.to_timedelta(df["lead_time_days"], unit="D")
print("CHECK | Check-Out rows where status date = arrival + nights:", (both & checkout & nights.notna() & (stay == nights)).sum(),
      "of", (both & checkout & nights.notna()).sum())
print("      among the conflicts: status date before arrival", (nights_diff & (df["reservation_status_date"] < df["arrival_date"])).sum(),
      "| before the booking date", (nights_diff & (df["reservation_status_date"] < booking_date)).sum(),
      "| before the last PMS update", (nights_diff & (df["reservation_status_date"] < df["record_updated_at"].dt.normalize())).sum())
print("CHECK | all rows: booking date after the last PMS update:", (booking_date > df["record_updated_at"].dt.normalize()).sum())

CHECK | Check-Out rows where status date = arrival + nights: 14838 of 14873
      among the conflicts: status date before arrival 30 | before the booking date 27 | before the last PMS update 30
CHECK | all rows: booking date after the last PMS update: 0


**Decision: keep the PMS nights, recompute the status date.** The price confirms the PMS nights in 31 of the 32 testable rows. The exception, BK308797, has a CRM price 41% above adr × nights: it is one of the inflated prices corrected in 6.1. The CRM status date is the wrong value: for a Check-Out it is the departure day, a rule that holds on 99.8% of the rows, while here it is often months before the arrival and even before the booking was made.

So the nights stay (flag `nights_conflict`) and the status date becomes arrival + nights (flag `status_date_from_pms`). It is derived from two values we have checked, not guessed. The CRM dates of these rows are not used to rebuild anything else.

In [19]:
df["nights_conflict"] = nights_diff
df["status_date_from_pms"] = nights_diff
df.loc[nights_diff, "reservation_status_date"] = df.loc[nights_diff, "arrival_date"] + pd.to_timedelta(nights[nights_diff], unit="D")
stay = (df["reservation_status_date"] - df["arrival_date"]).dt.days
print("nights_conflict:", df["nights_conflict"].sum(), "| status_date_from_pms:", df["status_date_from_pms"].sum(),
      "| Check-Out with status date != arrival + nights after:", (both & checkout & nights.notna() & (stay != nights)).sum())

nights_conflict: 35 | status_date_from_pms: 35 | Check-Out with status date != arrival + nights after: 0


### 5.5 CRM dates after the corrections
`date_inconsistent` (Phase A, CRM) marks the 88 rows where the CRM status date breaks the status rules. After 5.1-5.4 we check how many still break them.

In [20]:
st = df["reservation_status"]
sd = df["reservation_status_date"]
ar = df["arrival_date"]
still = ((st == "Check-Out") & (sd < ar)) | ((st == "Canceled") & (sd > ar)) | ((st == "No-Show") & (sd < ar))
flagged = df["date_inconsistent"]
print("date_inconsistent:", flagged.sum(), "| resolved:", (flagged & ~still).sum(),
      "| with status_from_pms:", (flagged & df["status_from_pms"]).sum(),
      "| status_date_from_pms:", (flagged & df["status_date_from_pms"]).sum(),
      "| arrival_conflict only:", (flagged & df["arrival_conflict"] & ~df["status_from_pms"] & ~df["status_date_from_pms"]).sum())
df.loc[flagged & still, ["booking_id", "reservation_status", "reservation_status_date", "arrival_date"]]

date_inconsistent: 88 | resolved: 87 | with status_from_pms: 50 | status_date_from_pms: 30 | arrival_conflict only: 7


,booking_id,reservation_status,reservation_status_date,arrival_date
18604,BK318605,No-Show,2016-03-28,2016-07-17


**Result.** `date_inconsistent` describes the raw CRM: 87 of its 88 rows are resolved by 5.1-5.4: 50 by `status_from_pms`, 30 by `status_date_from_pms`, and 7 only by the PMS arrival date (`arrival_conflict`). Only BK318605 (No-Show, status date 111 days before the arrival) still breaks the rule. The flag is kept as a record of the raw file: do not use it as a filter on the merged table.

## 6. Recompute the missing values

Prices and nights are linked by one formula, `total_price = adr × nights`, which holds within 1 euro on 22,436 of the 22,708 rows where all three are known and adr and nights are above zero. So when one of the three is missing, the other two give it back. Before using the formula we check the rows where it does not hold (6.1), because a wrong price would spread into the recomputed values. The check needs all three values: where the nights or the adr are missing, the CRM price cannot be tested (see 6.2 and 6.3).

### 6.1 Prices that do not match
272 rows break the formula. We look at the direction and size of the gap, and at the typical adr of the same hotel, room type, year and month, computed only on the rows that agree.

In [21]:
nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
testable = both & df["total_price"].notna() & (df["adr"] > 0) & (nights > 0)
price_diff = testable & ((df["total_price"] - df["adr"] * nights).abs() > 1)
ratio = df["total_price"] / (df["adr"] * nights)
print("testable rows:", testable.sum(), "| formula broken:", price_diff.sum())
print("CHECK | total_price / (adr x nights): min", ratio[price_diff].min().round(2), "max", ratio[price_diff].max().round(2),
      "| rows where the CRM price is lower:", (price_diff & (ratio < 1)).sum())

keys = ["hotel", "reserved_room_type", "arrival_date_year", "arrival_date_month"]
df["adr_agree"] = df["adr"].where(testable & ~price_diff)
typical = df.groupby(keys)["adr_agree"].transform("median")
df = df.drop(columns="adr_agree")
crm_adr = df["total_price"] / nights
print("CHECK | median of adr / typical adr | PMS adr:", (df["adr"] / typical)[price_diff].median().round(3),
      "| adr from the CRM price:", (crm_adr / typical)[price_diff].median().round(3),
      "| rows that agree (control):", (df["adr"] / typical)[testable & ~price_diff].median().round(3))
print("      PMS adr closer to the typical adr:", ((df["adr"] - typical).abs() < (crm_adr - typical).abs())[price_diff].sum(),
      "of", price_diff.sum())
cents = (df["total_price"] * 100).round() % 100 != 0
adr_dec = (df["adr"] * 100).round() % 100 != 0
print("CHECK | share with cents | CRM price: broken", cents[price_diff].mean().round(3), "agree", cents[testable & ~price_diff].mean().round(3),
      "| PMS adr: broken", adr_dec[price_diff].mean().round(3), "agree", adr_dec[testable & ~price_diff].mean().round(3))
gap = (df["total_price"] - df["adr"] * nights).abs()
print("CHECK | largest gap among the rows that agree:", gap[testable & ~price_diff].max().round(2),
      "| smallest gap among the broken ones:", gap[price_diff].min().round(2))

testable rows: 22708 | formula broken: 272
CHECK | total_price / (adr x nights): min 1.15 max 1.6 | rows where the CRM price is lower: 0
CHECK | median of adr / typical adr | PMS adr: 0.987 | adr from the CRM price: 1.344 | rows that agree (control): 1.0
      PMS adr closer to the typical adr: 200 of 272
CHECK | share with cents | CRM price: broken 0.996 agree 0.366 | PMS adr: broken 0.445 agree 0.428
CHECK | largest gap among the rows that agree: 0.0 | smallest gap among the broken ones: 4.72


**Decision: correct the CRM price.** The gap always goes the same way: the CRM price is 15% to 60% higher, never lower. A random typing error would go both ways. The PMS adr matches the typical price of the same room in the same month (median ratio 0.99, like the rows that agree), while the adr implied by the CRM price is 34% above it. The digits point the same way: almost every broken CRM price has cents, against about a third of the others, while the PMS adr of these rows has decimals as often as anywhere else. So the CRM price is inflated and the PMS adr is right: `total_price` becomes adr × nights, flag `price_inflated`. The 1-euro threshold does not matter: a row either matches to the cent or is at least 4.72 euro off (CHECK above).

In [22]:
df["price_inflated"] = price_diff
df.loc[price_diff, "total_price"] = (df["adr"] * nights)[price_diff].round(2)
print("price_inflated:", df["price_inflated"].sum(),
      "| formula broken after:", (testable & ((df["total_price"] - df["adr"] * nights).abs() > 1)).sum())

price_inflated: 272 | formula broken after: 0


### 6.2 Week nights
The 35 rows with `stays_in_week_nights` missing (code -1 in the raw PMS). Two sources, in order of reliability:
1. **CRM dates:** for Check-Out rows without a date problem, departure minus arrival gives the total nights; minus the weekend nights, the week nights. Flag `week_nights_from_crm`.
2. **Price:** `total_price / adr` gives the nights paid; minus the weekend nights, the week nights. Used only when the result is a whole number. It is an **ESTIMATE**: flag `week_nights_estimated`.

In [23]:
week_missing = df["in_pms"] & df["stays_in_week_nights"].isna()
stay = (df["reservation_status_date"] - df["arrival_date"]).dt.days
from_dates = week_missing & (df["reservation_status"] == "Check-Out") & ~df["date_inconsistent"] & stay.notna()
paid = df["total_price"] / df["adr"]
from_price = week_missing & ~from_dates & (df["adr"] > 0) & df["total_price"].notna() & ((paid - paid.round()).abs() < 0.01)
print("week nights missing:", week_missing.sum(), "| from the dates:", from_dates.sum(),
      "| from the price:", from_price.sum(), "| no source:", (week_missing & ~from_dates & ~from_price).sum())

week nights missing: 35 | from the dates: 18 | from the price: 15 | no source: 2


In [24]:
df["week_nights_from_crm"] = from_dates
df.loc[from_dates, "stays_in_week_nights"] = (stay - df["stays_in_weekend_nights"])[from_dates]
df["week_nights_estimated"] = from_price
df.loc[from_price, "stays_in_week_nights"] = (paid.round() - df["stays_in_weekend_nights"])[from_price]
print("rebuilt values:", df.loc[from_dates | from_price, "stays_in_week_nights"].value_counts().sort_index().to_dict())
print("week nights still missing:", df["stays_in_week_nights"][df["in_pms"]].isna().sum())

rebuilt values: {0.0: 1, 1.0: 10, 2.0: 11, 3.0: 4, 4.0: 3, 5.0: 2, 6.0: 2}
week nights still missing: 2


**CHECK of 6.1 on the rebuilt rows.** The rows whose week nights were rebuilt from the CRM dates could not be tested in 6.1, because their nights were missing. Now they can, so the 6.1 rule is applied to them too.

In [25]:
nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
late = df["week_nights_from_crm"] & df["total_price"].notna() & (df["adr"] > 0) & ((df["total_price"] - df["adr"] * nights).abs() > 1)
print("price_inflated found after 6.2:", late.sum(), "| booking:", df.loc[late, "booking_id"].tolist(),
      "| total_price / (adr x nights):", (df["total_price"] / (df["adr"] * nights))[late].round(3).tolist())
df["price_inflated"] = df["price_inflated"] | late
df.loc[late, "total_price"] = (df["adr"] * nights)[late].round(2)
print("price_inflated in total:", df["price_inflated"].sum())

price_inflated found after 6.2: 1 | booking: ['BK319532'] | total_price / (adr x nights): [1.165]
price_inflated in total: 273


### 6.3 adr
The adr missing in the PMS (set to NaN in Phase A because negative or above 1,000, or missing from the start) is recomputed as `total_price / nights`. Flag `adr_from_crm`. Limit: without the adr, 6.1 could not test these CRM prices. With the 6.1 rate (272 of 22,708, 1.2%), about 2 of the 177 could be inflated (ESTIMATE), and they cannot be found. For 0-night stays the formula cannot be used: every 0-night row with a known adr has adr 0, so these become 0, flag `adr_estimated` (ESTIMATE).

In [26]:
nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
adr_missing = df["in_pms"] & df["adr"].isna()
print("CHECK | adr of 0-night rows with a known adr:", df["adr"][nights == 0].value_counts().to_dict())
from_price = adr_missing & df["total_price"].notna() & (nights > 0)
zero = adr_missing & (nights == 0)
print("adr missing:", adr_missing.sum(), "| from the price:", from_price.sum(), "| 0 nights:", zero.sum(),
      "| no source:", (adr_missing & ~from_price & ~zero).sum())

CHECK | adr of 0-night rows with a known adr: {0.0: 145}
adr missing: 190 | from the price: 177 | 0 nights: 3 | no source: 10


In [27]:
df["adr_from_crm"] = from_price
df.loc[from_price, "adr"] = (df["total_price"] / nights)[from_price].round(2)
df["adr_estimated"] = zero
df.loc[zero, "adr"] = 0
print("adr still missing (PMS rows):", df["adr"][df["in_pms"]].isna().sum())

adr still missing (PMS rows): 10


### 6.4 total_price
The CRM price missing in rows that are also in the PMS is recomputed as adr × nights. Flag `price_from_pms`. The 500 PMS-only rows are left as they are: they have no CRM price at all, and the analyses can use adr.

In [28]:
price_missing = both & df["total_price"].isna()
from_adr = price_missing & df["adr"].notna() & nights.notna()
print("total_price missing:", price_missing.sum(), "| from adr x nights:", from_adr.sum())
df["price_from_pms"] = from_adr
df.loc[from_adr, "total_price"] = (df["adr"] * nights)[from_adr].round(2)
print("total_price still missing (rows in both files):", df["total_price"][both].isna().sum())

total_price missing: 469 | from adr x nights: 466
total_price still missing (rows in both files): 3


## 7. Other checks
Four checks that change no value. They describe limits of the data that matter for the analysis.

### 7.1 Weekend nights
The `info` sheet says weekend nights are the nights of Saturday and Sunday. From the arrival date and the nights we can count them ourselves and compare with `stays_in_weekend_nights`.

In [29]:
nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
known = df["in_pms"] & nights.notna()
day = df["arrival_date"].dt.dayofweek          # Monday = 0 ... Sunday = 6
sat_sun = 0
sun_mon = 0
for k in range(int(nights.max())):
    night_day = (day + k) % 7
    sat_sun = sat_sun + ((nights > k) & night_day.isin([5, 6]))
    sun_mon = sun_mon + ((nights > k) & night_day.isin([6, 0]))
print("rows:", known.sum())
print("CHECK | weekend nights = nights of Saturday and Sunday:", (known & (sat_sun == df["stays_in_weekend_nights"])).sum())
print("      weekend nights = nights of Sunday and Monday:", (known & (sun_mon == df["stays_in_weekend_nights"])).sum())

rows: 24248
CHECK | weekend nights = nights of Saturday and Sunday: 12078
      weekend nights = nights of Sunday and Monday: 24248


**Known limit, not corrected.** The count matches the `info` definition in only half of the rows, but it matches in every row if the weekend nights are those starting on Sunday and Monday: the dates and the weekend count are one day apart. Either all arrival dates are one day late, or the system counts the weekend differently from the dictionary.

We cannot tell which. The CRM has the same check-in dates, and its departure dates agree with them, while no column (week number, weekday) can decide. Shifting 25,000 dates or rewriting the weekend count of about 12,170 rows would be a guess. Nothing changes; in the analysis we do not draw conclusions by day of the week or on weekend against week nights.

### 7.2 Repeated guests
`is_repeated_guest` should agree with the guest's history in the CRM (`previous_cancellations`, `previous_bookings_not_canceled`).

In [30]:
crm_rows = df["in_crm"]
repeated = df["is_repeated_guest"]
stays_before = df["previous_bookings_not_canceled"]
cancels_before = df["previous_cancellations"]
print("CHECK | not repeated, only past cancellations:", (crm_rows & (repeated == 0) & (stays_before == 0) & (cancels_before > 0)).sum())
print("      not repeated, but past stays:", (crm_rows & (repeated == 0) & (stays_before > 0)).sum())
print("      repeated, no history at all:", (crm_rows & (repeated == 1) & (stays_before == 0) & (cancels_before == 0)).sum())
print("      repeated, only past cancellations:", (crm_rows & (repeated == 1) & (stays_before == 0) & (cancels_before > 0)).sum())

CHECK | not repeated, only past cancellations: 1102
      not repeated, but past stays: 152
      repeated, no history at all: 114
      repeated, only past cancellations: 74


**Decision: flag only.** A guest who only cancelled before has never stayed, so "not repeated" is coherent. Two cases are real contradictions: not repeated with past stays, and repeated with no history at all. Flag `repeated_inconsistent`. No value changes, because nothing shows which of the columns is wrong. The ambiguous case (repeated, only past cancellations) is left as it is.

In [31]:
df["repeated_inconsistent"] = crm_rows & (((repeated == 0) & (stays_before > 0)) |
                                         ((repeated == 1) & (stays_before == 0) & (cancels_before == 0)))
print("repeated_inconsistent:", df["repeated_inconsistent"].sum())

repeated_inconsistent: 266


### 7.3 Repeated status dates
Some status dates are shared by many bookings, for example 1 January 2015. A date like that can be a code for "unknown".

In [32]:
print("most frequent status dates:", df["reservation_status_date"].value_counts().head(5).to_dict())
jan1 = df["reservation_status_date"] == "2015-01-01"
booking_date = df["arrival_date"] - pd.to_timedelta(df["lead_time_days"], unit="D")
print("CHECK | 2015-01-01:", jan1.sum(), "| status:", df["reservation_status"][jan1].value_counts().to_dict(),
      "| segment:", df["market_segment"][jan1].value_counts().to_dict(), "| hotel:", df["hotel"][jan1].value_counts().to_dict())
print("      booking date:", booking_date[jan1].value_counts().to_dict())
oct21 = df["reservation_status_date"] == "2015-10-21"
print("CHECK | 2015-10-21:", oct21.sum(), "| cancelled:", (oct21 & (df["reservation_status"] == "Canceled")).sum(),
      "| different booking dates:", booking_date[oct21].nunique())

most frequent status dates: {Timestamp('2015-10-21 00:00:00'): 298, Timestamp('2016-11-25 00:00:00'): 172, Timestamp('2015-07-06 00:00:00'): 160, Timestamp('2015-01-01 00:00:00'): 151, Timestamp('2016-01-18 00:00:00'): 136}
CHECK | 2015-01-01: 151 | status: {'Canceled': 151} | segment: {'Groups': 151} | hotel: {'City Hotel': 151}
      booking date: {Timestamp('2014-10-17 00:00:00'): 151}
CHECK | 2015-10-21: 298 | cancelled: 280 | different booking dates: 16


**Decision: nothing to change.** The 151 rows are all cancelled Groups of the City Hotel, booked on the same day: one group block cancelled at once, not a code for "unknown". The most frequent date, 2015-10-21, is also mostly cancellations, but booked on 16 different days: a frequent status date is a day with many cancellations, not always one block.

### 7.4 Bookings identical except the code
Two different booking codes with the same values in every other column could be one booking recorded twice.

In [33]:
same = [c for c in df.columns if c != "booking_id"]
near = df[df.duplicated(subset=same, keep=False)]
print("rows equal to an earlier row except booking_id:", df.duplicated(subset=same).sum(),
      "| rows in these groups:", len(near))
print("CHECK | status:", near["reservation_status"].value_counts().to_dict(),
      "| hotel:", near["hotel"].value_counts().to_dict())
print("      segment:", near["market_segment"].value_counts().to_dict(),
      "| deposit:", near["deposit_type"].value_counts().to_dict())

rows equal to an earlier row except booking_id: 87 | rows in these groups: 169
CHECK | status: {'Canceled': 154, 'Check-Out': 15} | hotel: {'City Hotel': 155, 'Resort Hotel': 14}
      segment: {'Groups': 91, 'Offline TA/TO': 66, 'Corporate': 8, 'Online TA': 4} | deposit: {'Non Refund': 117, 'No Deposit': 22}


**Decision: keep them.** 87 rows are equal to an earlier row in every column except the code (169 rows in groups of 2 or 3). 154 of the 169 are cancelled, almost all are Groups (91) or Offline TA/TO (66), and 117 of the 139 with a deposit type are Non Refund. This is the profile of group blocks, several rooms booked and cancelled together, not of a booking recorded twice. It is an inference: no column proves it, so no row is removed. Bookings repeated with the same code were already removed in Phase A.

## 8. Export
We drop the columns that are now duplicates:
- `property`: identical to `hotel` in every row in both files, and already used to fill it;
- `checkin_date`: already used to fill `arrival_date`; where the two differ, the CRM date was shown to be wrong (5.1).

`n_adults` stays: in the 56 `adults_conflict` rows nothing proves which file is right, so the CRM value is kept next to the PMS one. Final order by booking code, then the shape and how many rows each flag marks.

In [34]:
print("rows, columns before the drop:", df.shape)
df = df.drop(columns=["property", "checkin_date"])
df = df.sort_values("booking_id").reset_index(drop=True)
print("rows, columns:", df.shape, "| booking_id unique:", df["booking_id"].is_unique)
flags = ["arrival_conflict", "adults_from_crm", "adults_conflict", "adults_unverified", "status_from_pms",
         "nights_conflict", "price_inflated", "week_nights_from_crm", "week_nights_estimated",
         "adr_from_crm", "adr_estimated", "price_from_pms", "status_date_from_pms", "repeated_inconsistent"]
df[flags].sum()

rows, columns before the drop: (25000, 70)
rows, columns: (25000, 68) | booking_id unique: True


arrival_conflict          45
adults_from_crm           19
adults_conflict           56
adults_unverified          2
status_from_pms           50
nights_conflict           35
price_inflated           273
week_nights_from_crm      18
week_nights_estimated     15
adr_from_crm             177
adr_estimated              3
price_from_pms           466
status_date_from_pms      35
repeated_inconsistent    266
dtype: int64

In [35]:
df.to_csv("../data/processed/merged_clean.csv", index=False)
print("saved ../data/processed/merged_clean.csv:", df.shape)

saved ../data/processed/merged_clean.csv: (25000, 68)
